# MDN label-flip diagnostic

Measures how much the trained `BiggWithMDNFeats` per-feature mixture changes when the input label_embed is flipped, holding `h` and `z` constant. Compares tolokers (works) vs weibo (partial) vs questions (broken) to decide whether class-weighted loss is the right next move.

**Decision rule** on `ratio = median(W1 questions) / median(W1 tolokers)`:
- `< 0.25`: MDN head ignores `label_embed` on questions → class-weight the cont loss.
- `> 0.75`: MDN uses `label_embed` equivalently → label-head / label-sampling is the problem.
- otherwise: both factors contribute.

Requires `model.pt` per dataset, produced by retraining with `SAVE_MODEL=true`. The trajectory section additionally requires `SAVE_MODEL_EVERY > 0` so per-epoch snapshots (`model_epoch{N}.pt`) are written. See plan at `~/.claude/plans/write-the-diagnostic-plan-resilient-scone.md`.

In [ ]:
import os, sys, pickle, glob, re
from types import SimpleNamespace
import numpy as np
import torch
import networkx as nx
from dgl import load_graphs
import matplotlib.pyplot as plt

ROOT = '/home/eirik/progg/master/SynBench'
sys.path.insert(0, os.path.join(ROOT, 'bigg'))
sys.path.insert(0, os.path.join(ROOT, 'experiments', 'subsample_search'))

from bigg.extension.customized_models import BiggWithMDNFeats
from bigg.extension.preprocessing import apply_normalization, bfs_reorder
from bigg.model.tree_clib.tree_lib import setup_treelib, TreeLib
from splitsource import load_split_source

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RESULTS = os.path.join(ROOT, 'experiments', 'diagnostics', 'results')
os.makedirs(RESULTS, exist_ok=True)
print(f'Device: {DEVICE}')

In [ ]:
# Per-dataset config. save_name matches the directory the pipeline writes under
# datasets/synthetic/bigg/{dataset}/hidden_labels/. model.pt (and any model_epoch{N}.pt) live inside it.
RUNS = {
    'tolokers': {
        'save_name': 'blksize_-1_b_1_lr_0.0001_epochs_300_noise_0.3_ss_0.3_norm_cdf_bfs_lw_0.01_0.1_det_masked_binfeat_vae16_kl0.25_lnmdn8_lsf-10.0_loadsub_metis_K15_split0_n15',
    },
    'weibo': {
        'save_name': 'blksize_-1_b_1_lr_0.0003_epochs_300_noise_0.3_ss_0.3_norm_cdf_bfs_lw_0.01_0.1_det_masked_binfeat_vae16_kl0.25_lnmdn8_lsf-4.0_loadsub_metis_K15_split0_n15',
    },
    'questions': {
        'save_name': 'blksize_-1_b_1_lr_0.0003_epochs_300_noise_0.3_ss_0.3_norm_cdf_bfs_lw_0.01_0.1_det_masked_binfeat_vae16_kl0.25_lnmdn8_lsf-4.0_loadsub_metis_K15_split0_n15',
    },
}

def run_dir(dataset):
    return os.path.join(ROOT, 'datasets', 'synthetic', 'bigg', dataset,
                        'hidden_labels', RUNS[dataset]['save_name'])

for ds in RUNS:
    rd = run_dir(ds)
    final = os.path.join(rd, 'model.pt')
    snaps = sorted(glob.glob(os.path.join(rd, 'model_epoch*.pt')),
                   key=lambda p: int(re.search(r'epoch(\d+)\.pt', p).group(1)))
    snap_eps = [int(re.search(r'epoch(\d+)\.pt', p).group(1)) for p in snaps]
    print(f'{ds:10s} model.pt: {"OK" if os.path.exists(final) else "MISSING"} | '
          f'snapshots: {snap_eps if snap_eps else "(none)"}')

In [ ]:
def load_trained_model(model_pt_path):
    """Reconstruct BiggWithMDNFeats and load state_dict from a saved model.pt."""
    blob = torch.load(model_pt_path, map_location='cpu', weights_only=False)
    cmd_args = SimpleNamespace(**blob['cmd_args'])
    pa = SimpleNamespace(**blob['pipeline_args'])
    cmd_args.device = DEVICE
    setup_treelib(cmd_args)
    model = BiggWithMDNFeats(
        cmd_args, feat_dim=blob['feat_dim'], num_classes=blob['num_classes'],
        n_components=pa.mdn_components, label_temp=pa.label_temp,
        noise_std=pa.noise_std, logsigma_floor=pa.mdn_logsigma_floor,
        binary_feat=pa.binary_feat, binary_idx=blob['binary_idx'],
        vae_feat=pa.vae_feat, vae_dim=pa.vae_dim, kl_weight=pa.kl_weight,
        logvar_floor=pa.logvar_floor, mdn_base=pa.mdn_base,
    ).to(DEVICE)
    model.load_state_dict(blob['state_dict'])
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)
    assert not model.training
    return model, cmd_args, pa, blob['feat_dim'], blob['binary_idx']

In [ ]:
def load_subgraphs_for_inference(dataset, cmd_args, pa, feat_dim, binary_idx):
    """Replicate pipeline.py's load_subsamples path. Returns the list of
    (gid, sub_nd_device, sub_label_mask, sub_num_nodes) tuples the pipeline trained on,
    plus a flat tensor of labels in the post-BFS row order the MDN head will see."""
    # Full graph + features + labels
    graphs, _ = load_graphs(os.path.join(ROOT, 'datasets', 'original', dataset))
    graph = graphs[0]
    cont_feats = graph.ndata['feature']
    labels = graph.ndata['label']

    # Re-apply saved normalization (cont columns only; binary cols left raw)
    norm_stats = torch.load(os.path.join(run_dir(dataset), 'norm_stats.pt'),
                            map_location='cpu', weights_only=False)
    if binary_idx:
        cont_idx = sorted(set(range(cont_feats.shape[1])) - set(binary_idx))
        normed = apply_normalization(cont_feats[:, cont_idx], norm_stats)
        cont_feats = cont_feats.clone()
        cont_feats[:, cont_idx] = normed
    else:
        cont_feats = apply_normalization(cont_feats, norm_stats)

    node_data = torch.cat([cont_feats, labels.unsqueeze(1).float()], dim=1).to(DEVICE)

    label_mask = None
    if pa.mask_test_labels:
        train_m = graph.ndata['train_masks'][:, 0].bool()
        val_m = graph.ndata['val_masks'][:, 0].bool()
        label_mask = (train_m | val_m).to(DEVICE)

    # Load the persisted subsamples
    pkl_path = os.path.join(ROOT, 'datasets', 'bigg_subsamples', dataset,
                            pa.subsampling_config, f'split{pa.split_id}.pkl')
    with open(pkl_path, 'rb') as f:
        payload = pickle.load(f)
    src = load_split_source(dataset, pa.split_id,
                            data_dir=os.path.join(ROOT, 'datasets', 'original'))
    orig_ids_full = src.orig_ids.long()

    subgraphs = []
    all_labels_post_bfs = []  # labels in the order the MDN head sees them
    cmd_args.max_num_nodes = 0
    for sg_nx, _stored_sub_nd, orig_idx_lcc in payload['partitions']:
        full_idx = orig_ids_full[orig_idx_lcc.long()]
        sub_nd = node_data[full_idx.to(node_data.device)]
        if sg_nx.is_multigraph():
            sg_simple = nx.Graph()
            sg_simple.add_nodes_from(sg_nx.nodes())
            sg_simple.add_edges_from(sg_nx.edges())
            sg_nx = sg_simple

        if pa.bfs_preprocess:
            sg_nx, sub_nd, perm = bfs_reorder(sg_nx, sub_nd)
            sub_lm = label_mask[full_idx[perm]] if label_mask is not None else None
        else:
            sub_lm = label_mask[full_idx] if label_mask is not None else None

        gid = TreeLib.InsertGraph(sg_nx)
        n = sg_nx.number_of_nodes()
        cmd_args.max_num_nodes = max(cmd_args.max_num_nodes, n)
        sub_nd_dev = sub_nd.to(DEVICE)
        subgraphs.append((gid, sub_nd_dev, sub_lm, n))
        all_labels_post_bfs.append(sub_nd_dev[:, feat_dim].long().cpu())

    labels_flat = torch.cat(all_labels_post_bfs, dim=0)
    cmd_args.has_node_feats = True
    return subgraphs, labels_flat

In [ ]:
def capture_h_cond(model, subgraphs):
    """Run model.forward_train per subgraph with a hook on mdn_head to capture h_cond.
    Returns a single (N_total, 2*embed_dim + vae_dim) tensor in row-order across subgraphs."""
    captured = []

    def hook(_module, inputs):
        captured.append(inputs[0].detach().cpu())

    handle = model.mdn_head.register_forward_pre_hook(hook)
    try:
        with torch.no_grad():
            for gid, sub_nd, sub_lm, _n in subgraphs:
                _ = model.forward_train([gid], node_feats=sub_nd, label_mask=sub_lm)
    finally:
        handle.remove()
    return torch.cat(captured, dim=0)

In [ ]:
def mdn_w1_under_flip(model, h_cond_all, labels_flat, embed_dim, *, S=512, chunk=1024):
    """Per-(node, feature) Wasserstein-1 in logit space between MDN(true label) and
    MDN(flipped label). Holds h, z constant; only label_embed changes.
    Also computes an identity-control W1 (two independent draws from the true MDN) as a noise floor."""
    E = embed_dim
    N = h_cond_all.shape[0]
    labels_dev = labels_flat.to(DEVICE)

    # Slicing sanity: h_cond[:, E:2E] should equal nodelabel_encoding(labels)
    label_em_captured = h_cond_all[:, E:2*E].to(DEVICE)
    label_em_expected = model.nodelabel_encoding(labels_dev)
    max_abs_diff = (label_em_captured - label_em_expected).abs().max().item()
    assert max_abs_diff < 1e-5, (
        f'Captured label_em slice does not match nodelabel_encoding(labels) '
        f'(max abs diff {max_abs_diff:.2e}) — slicing/label-order is wrong.')

    flip_em_all = model.nodelabel_encoding(1 - labels_dev).cpu()

    F = model.cont_feat_dim
    w1_real = torch.empty(N, F)
    w1_ident = torch.empty(N, F)

    def _sample_sorted(log_pi, mu, log_sigma, S):
        pi = log_pi.exp()
        Nc, Fc, K = pi.shape
        flat_pi = pi.reshape(Nc * Fc, K)
        comp = torch.multinomial(flat_pi, S, replacement=True).view(Nc, Fc, S)
        s = mu.gather(-1, comp) + log_sigma.gather(-1, comp).exp() * torch.randn(
            Nc, Fc, S, device=mu.device)
        return s.sort(-1).values

    with torch.no_grad():
        for start in range(0, N, chunk):
            end = min(start + chunk, N)
            h_chunk = h_cond_all[start:end].to(DEVICE)
            flip_chunk = flip_em_all[start:end].to(DEVICE)
            h = h_chunk[:, :E]
            z_part = h_chunk[:, 2*E:]
            h_cond_true = h_chunk  # exactly what the head saw at training time
            h_cond_flip = torch.cat([h, flip_chunk, z_part], dim=-1)

            lp_t, mu_t, ls_t = model._mdn_params(h_cond_true)
            lp_f, mu_f, ls_f = model._mdn_params(h_cond_flip)

            s_t1 = _sample_sorted(lp_t, mu_t, ls_t, S)
            s_f  = _sample_sorted(lp_f, mu_f, ls_f, S)
            w1_real[start:end] = (s_t1 - s_f).abs().mean(-1).cpu()

            # Identity control: two independent draws from the true MDN.
            s_t2 = _sample_sorted(lp_t, mu_t, ls_t, S)
            w1_ident[start:end] = (s_t1 - s_t2).abs().mean(-1).cpu()

    return w1_real.numpy(), w1_ident.numpy()

In [ ]:
def _npz_tag_for_ckpt(ckpt_filename):
    """model.pt -> '' (-> {ds}_w1.npz); model_epoch50.pt -> '_epoch50' (-> {ds}_w1_epoch50.npz)."""
    m = re.match(r'model(_epoch\d+)?\.pt$', ckpt_filename)
    return m.group(1) or '' if m else '_' + ckpt_filename.replace('.pt', '')

def run_dataset(dataset, *, ckpt_filename='model.pt', S=512, chunk=1024, save_npz=True):
    model_pt = os.path.join(run_dir(dataset), ckpt_filename)
    print(f'\n=== {dataset} ({ckpt_filename}) ===')
    print(f'Loading {model_pt}')
    model, cmd_args, pa, feat_dim, binary_idx = load_trained_model(model_pt)
    subgraphs, labels_flat = load_subgraphs_for_inference(
        dataset, cmd_args, pa, feat_dim, binary_idx)
    print(f'Subgraphs: {len(subgraphs)}; nodes per: {[n for _,_,_,n in subgraphs]}')
    print(f'Total nodes: {labels_flat.numel()}; anomaly fraction: {labels_flat.float().mean():.4f}')

    h_cond = capture_h_cond(model, subgraphs)
    print(f'Captured h_cond shape: {tuple(h_cond.shape)} '
          f'(expect 2*embed_dim + vae_dim = {2*cmd_args.embed_dim + (pa.vae_dim if pa.vae_feat else 0)})')

    w1, w1_ident = mdn_w1_under_flip(
        model, h_cond, labels_flat, cmd_args.embed_dim, S=S, chunk=chunk)

    if save_npz:
        out_path = os.path.join(RESULTS, f'{dataset}_w1{_npz_tag_for_ckpt(ckpt_filename)}.npz')
        np.savez(out_path,
                 w1=w1, w1_ident=w1_ident, labels=labels_flat.numpy(),
                 lsf=pa.mdn_logsigma_floor, n_components=pa.mdn_components,
                 embed_dim=cmd_args.embed_dim, dataset=dataset, ckpt=ckpt_filename)
        print(f'Saved {out_path}')
    print(f'  median W1 (flip)     = {np.median(w1):.4f}')
    print(f'  median W1 (identity) = {np.median(w1_ident):.4f}')
    print(f'  per-node W1 (flip), label=0 mean = {w1[labels_flat.numpy()==0].mean():.4f}, '
          f'label=1 mean = {w1[labels_flat.numpy()==1].mean():.4f}')
    return w1, w1_ident, labels_flat.numpy()

## Final-checkpoint cross-dataset comparison

Runs the diagnostic on `model.pt` (end of training) for each dataset, then renders the cross-dataset violin and verdict. Heavy: each dataset takes a few minutes.

In [ ]:
tolokers_w1, tolokers_id, tolokers_lab = run_dataset('tolokers')

In [ ]:
weibo_w1, weibo_id, weibo_lab = run_dataset('weibo')

In [ ]:
questions_w1, questions_id, questions_lab = run_dataset('questions')

In [ ]:
# Cross-dataset comparison: per-node mean W1, split by label group.
def load_npz(dataset, tag=''):
    z = np.load(os.path.join(RESULTS, f'{dataset}_w1{tag}.npz'), allow_pickle=True)
    return z['w1'], z['w1_ident'], z['labels']

groups = []  # list of (label, per_node_mean_w1)
ident_floor = []
for ds in ['tolokers', 'weibo', 'questions']:
    w1, w1_id, lab = load_npz(ds)
    per_node = w1.mean(axis=1)
    ident_floor.append(w1_id.mean())
    groups.append((f'{ds}\nlabel=0', per_node[lab == 0]))
    groups.append((f'{ds}\nlabel=1', per_node[lab == 1]))

fig, ax = plt.subplots(figsize=(10, 5))
ax.violinplot([g[1] for g in groups], showmedians=True)
ax.set_xticks(range(1, len(groups) + 1))
ax.set_xticklabels([g[0] for g in groups])
ax.axhline(np.mean(ident_floor), color='gray', linestyle='--', linewidth=1,
           label=f'mean identity-control noise floor = {np.mean(ident_floor):.4f}')
ax.set_ylabel('per-node mean W1 (logit space)')
ax.set_title('MDN sensitivity to label_embed flip — by dataset × class')
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
# Verdict
med = {}
for ds in ['tolokers', 'weibo', 'questions']:
    w1, _, _ = load_npz(ds)
    med[ds] = float(np.median(w1))
ratio_q_t = med['questions'] / med['tolokers'] if med['tolokers'] > 0 else float('nan')
ratio_w_t = med['weibo']     / med['tolokers'] if med['tolokers'] > 0 else float('nan')
print(f'median W1 (flip) — tolokers: {med["tolokers"]:.4f}, '
      f'weibo: {med["weibo"]:.4f}, questions: {med["questions"]:.4f}')
print(f'ratio_q_t = {ratio_q_t:.3f}; ratio_w_t = {ratio_w_t:.3f}')
if ratio_q_t < 0.25:
    verdict = 'MDN head IGNORES label_embed on questions — ship class weighting first.'
elif ratio_q_t > 0.75:
    verdict = 'MDN head USES label_embed equivalently — investigate label head / sampled-label path.'
else:
    verdict = 'Mixed: partial label_embed usage; both factors likely contribute.'
print(f'\nVERDICT: {verdict}')

print('\nNote: tolokers used logsigma_floor=-10.0; weibo/questions used -4.0. '
      'Absolute W1 magnitudes are not directly comparable across that floor; '
      'the diagnostic answers the relative question (does the head respond to a flip?), '
      'which the ratio captures.')

## Training trajectory — overfitting check

Iterates `model_epoch{N}.pt` snapshots (requires `SAVE_MODEL_EVERY > 0` at training) and plots median W1 vs epoch per dataset. If label-embed sensitivity *decreases* over training, that's direct evidence the head learns to ignore the label as training progresses — supporting the exposure-bias / overfitting hypothesis.

Heavy: ~1-5 min per snapshot per dataset. Comment out datasets you don't want.

In [ ]:
def trajectory_compute(dataset, save_npz=True):
    """Run the W1 diagnostic for every model_epoch{N}.pt snapshot in run_dir(dataset).
    Returns list of (epoch, w1, w1_ident, labels) sorted by epoch."""
    ckpts = sorted(glob.glob(os.path.join(run_dir(dataset), 'model_epoch*.pt')),
                   key=lambda p: int(re.search(r'epoch(\d+)\.pt', p).group(1)))
    if not ckpts:
        print(f'{dataset}: no model_epoch*.pt snapshots — retrain with SAVE_MODEL_EVERY > 0.')
        return []
    out = []
    for path in ckpts:
        ep = int(re.search(r'epoch(\d+)\.pt', path).group(1))
        w1, w1_id, lab = run_dataset(dataset, ckpt_filename=os.path.basename(path),
                                     save_npz=save_npz)
        out.append((ep, w1, w1_id, lab))
    return out

# Run (heavy). Comment out datasets you don't want.
trajs = {}
for ds in ['tolokers', 'weibo', 'questions']:
    trajs[ds] = trajectory_compute(ds)

In [ ]:
# Plot median W1 vs epoch per dataset. Re-runnable from the saved .npz files
# without re-running trajectory_compute (uncomment loader below to skip compute).
#
# trajs = {}
# for ds in ['tolokers', 'weibo', 'questions']:
#     files = sorted(glob.glob(os.path.join(RESULTS, f'{ds}_w1_epoch*.npz')),
#                    key=lambda p: int(re.search(r'epoch(\d+)\.npz', p).group(1)))
#     entries = []
#     for f in files:
#         ep = int(re.search(r'epoch(\d+)\.npz', f).group(1))
#         z = np.load(f, allow_pickle=True)
#         entries.append((ep, z['w1'], z['w1_ident'], z['labels']))
#     trajs[ds] = entries

fig, ax = plt.subplots(figsize=(8, 5))
for ds, traj in trajs.items():
    if not traj:
        continue
    eps  = [t[0] for t in traj]
    meds = [float(np.median(t[1])) for t in traj]
    ids  = [float(np.median(t[2])) for t in traj]
    ax.plot(eps, meds, marker='o', label=f'{ds} (flip)')
    ax.plot(eps, ids,  marker='x', linestyle='--', alpha=0.5,
            label=f'{ds} (identity floor)')
ax.set_xlabel('Epoch')
ax.set_ylabel('median per-(node,feature) W1 (logit space)')
ax.set_title('MDN label-embed sensitivity over training')
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()